# 05 — Autoencoder convolucional (deteccion de anomalias, ULB)
Entrena solo con transacciones legitimas y detecta fraude por error de reconstruccion.
Umbral afinado con busqueda amplia. Requiere el 01. Corre todas las celdas.

In [ ]:
import os, datetime as _dt

def iniciar_ejecucion(nombre, config=None, base="resultados"):
    os.makedirs(base, exist_ok=True)
    nums = []
    for d in os.listdir(base):
        if d.startswith("ejecucion_") and os.path.isdir(os.path.join(base, d)):
            try: nums.append(int(d.split("_")[1]))
            except: pass
    n = max(nums)+1 if nums else 1
    carpeta = os.path.join(base, f"ejecucion_{n}")
    os.makedirs(carpeta)
    with open(os.path.join(carpeta, "config.txt"), "w", encoding="utf-8") as f:
        f.write("notebook : " + str(nombre) + "\n")
        f.write("fecha    : " + _dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S") + "\n")
        if config:
            for k, v in config.items():
                f.write(str(k) + " : " + str(v) + "\n")
    print("[Ejecucion " + str(n) + "] Resultados en: " + carpeta)
    return carpeta

def log(texto, carpeta):
    print(texto)
    with open(os.path.join(carpeta, "resultados.txt"), "a", encoding="utf-8") as f:
        f.write(str(texto) + "\n")

def guardar_csv(resultados, ruta):
    if not resultados: return
    cols = list(resultados[0].keys())
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(",".join(cols) + "\n")
        for r in resultados:
            f.write(",".join(str(r[c]) for c in cols) + "\n")


In [ ]:
import numpy as np, os, time
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import (recall_score,f1_score,roc_auc_score,average_precision_score,precision_recall_curve)
import matplotlib.pyplot as plt
gpus=tf.config.list_physical_devices("GPU")
for g in gpus: tf.config.experimental.set_memory_growth(g,True)
print(f"GPUs: {len(gpus)}" if gpus else "Sin GPU, se usara CPU")
DATOS="datos_procesados/ulb.npz"; CORRIDAS=5; EPOCAS=30; BATCH=256

In [ ]:
def construir(nf):
    e=layers.Input(shape=(nf,1))
    x=layers.Conv1D(32,3,activation="relu",padding="same")(e)
    x=layers.MaxPooling1D(2,padding="same")(x)
    x=layers.Conv1D(16,3,activation="relu",padding="same")(x)
    x=layers.UpSampling1D(2)(x)
    x=layers.Conv1D(32,3,activation="relu",padding="same")(x)
    s=layers.Conv1D(1,3,activation="linear",padding="same")(x)
    m=models.Model(e,s); m.compile(optimizer="adam",loss="mse"); return m

def err_recon(m,X):
    Xr=X.reshape(-1,X.shape[1],1); rec=m.predict(Xr,batch_size=4096,verbose=0)
    L=min(Xr.shape[1],rec.shape[1]); return np.mean((Xr[:,:L,:]-rec[:,:L,:])**2,axis=(1,2))

def umbral_opt(err_legit,err_val,y_val):
    lo,hi=np.percentile(err_legit,50),np.percentile(err_val,99.9)
    cand=np.linspace(lo,hi,60); bu,bf=cand[0],-1
    for u in cand:
        f=f1_score(y_val,(err_val>u).astype(int),zero_division=0)
        if f>bf: bu,bf=u,f
    return bu

In [ ]:
carpeta=iniciar_ejecucion("05_autoencoder",{"datos":DATOS,"corridas":CORRIDAS,"epocas":EPOCAS})
d=np.load(DATOS,allow_pickle=True)
Xtr,ytr=d["X_train"],d["y_train"]; Xte,yte=d["X_test"],d["y_test"]; nf=Xtr.shape[1]
Xleg=Xtr[ytr==0]
log(f"Entrenamiento (solo legitimas): {Xleg.shape} | Test: {Xte.shape} fraude={int(yte.sum())}",carpeta)
metricas=[]
for c in range(CORRIDAS):
    tf.keras.utils.set_random_seed(42+c)
    m=construir(nf)
    es=tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=4,restore_best_weights=True)
    t0=time.perf_counter()
    m.fit(Xleg.reshape(-1,nf,1),Xleg.reshape(-1,nf,1),epochs=EPOCAS,batch_size=BATCH,validation_split=0.1,callbacks=[es],verbose=2 if c==0 else 0)
    t_ent=time.perf_counter()-t0
    u=umbral_opt(err_recon(m,Xleg),err_recon(m,Xtr),ytr)
    et=err_recon(m,Xte); pred=(et>u).astype(int)
    mm={"umbral":u,"recall":recall_score(yte,pred,zero_division=0),"f1":f1_score(yte,pred,zero_division=0),
        "auc_roc":roc_auc_score(yte,et),"auc_pr":average_precision_score(yte,et),"tiempo_s":t_ent}
    metricas.append(mm)
    log(f"Corrida {c+1}/{CORRIDAS}: "+"  ".join(f"{k}={v:.4f}" for k,v in mm.items()),carpeta)
    if c==0:
        lim=np.percentile(et,99)
        plt.figure(figsize=(8,4))
        plt.hist(et[yte==0],bins=80,range=(0,lim),alpha=0.6,label="Legitimas",density=True)
        plt.hist(et[yte==1],bins=80,range=(0,lim),alpha=0.6,label="Fraude",density=True)
        plt.axvline(u,color="red",linestyle="--",label="Umbral")
        plt.xlabel("Error de reconstruccion"); plt.ylabel("Densidad"); plt.legend(); plt.title("Errores (ULB)")
        plt.savefig(os.path.join(carpeta,"hist_errores.png"),dpi=150,bbox_inches="tight"); plt.show()
        pr,rc,_=precision_recall_curve(yte,et)
        plt.figure(figsize=(7,5)); plt.plot(rc,pr,lw=2,label=f"AUC-PR={mm['auc_pr']:.3f}")
        plt.xlabel("Recall"); plt.ylabel("Precision"); plt.legend(); plt.grid(alpha=0.3); plt.title("Curva PR - Autoencoder (ULB)")
        plt.savefig(os.path.join(carpeta,"curva_pr.png"),dpi=150,bbox_inches="tight"); plt.show()
log(f"\n=== RESUMEN (media +/- desv, {CORRIDAS} corridas) - Autoencoder ULB ===",carpeta)
for k in metricas[0]:
    v=[x[k] for x in metricas]; log(f"{k:10s}: {np.mean(v):.4f} +/- {np.std(v):.4f}",carpeta)
guardar_csv(metricas,os.path.join(carpeta,"metricas_por_corrida.csv"))
log(f"\nGuardado en: {carpeta}",carpeta)